In [14]:
%pip install seaborn


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

In [16]:
file_path = "CSI_7_MAL_2324_CW_resit_data.xlsx"

diagnosis_df = pd.read_excel(
    file_path,
    sheet_name="Diagnosis target"
)

cognitive_scores_df = pd.read_excel(
    file_path,
    sheet_name="Cognitive score"
)

data_df = pd.read_excel(
    file_path,
    sheet_name="Data"
)

print("Diagnosis data:", diagnosis_df.shape)
print("Cognitive data:", cognitive_scores_df.shape)
print("Patient data:", data_df.shape)

Diagnosis data: (1737, 4)
Cognitive data: (1737, 10)
Patient data: (1737, 361)


In [17]:
print("Diagnosis columns:")
print(diagnosis_df.columns.tolist())

print("\nCognitive score columns:")
print(cognitive_scores_df.columns.tolist())

print("\nPatient data columns:")
print(data_df.columns.tolist())

display(diagnosis_df.head())
display(cognitive_scores_df.head())
display(data_df.head())

Diagnosis columns:
['RID', 'Test_data', 'Diagnosis', 'FDG_PET']

Cognitive score columns:
['RID', 'Test_data', 'CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting']

Patient data columns:
['RID', 'Test_data', 'Age', 'Gender', 'Year_education', 'Ethnicity', 'Race', 'Marital_status', 'High_risk_ApoE4', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'Intra cranial volume', 'Volume (WM Parcellation) of RightPallidum', 'Volume (Cortical Parcellation) of RightParacentral', 'Surface Area of RightParacentral', 'Cortical Thickness Average of RightParacentral', 'Cortical Thickness Standard Deviation of RightParacentral', 'Volume (Cortical Parcellation) of RightParahippocampal', 'Surface Area of RightParahippocampal', 'Cortical Thickness Average of RightParahippocampal', 'Cortical Thickness Standard Deviation of RightParahippocampal', 'Volume (Cortical Parcellation) of RightParsOpercularis', 'Surface

,RID,Test_data,Diagnosis,FDG_PET
0,2,0,CN,1.36926
1,3,0,AD,1.09079
2,4,1,LMCI,NaN
3,5,0,CN,1.29799
4,6,0,LMCI,NaN


,RID,Test_data,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,RAVLT_perc_forgetting
0,2,0,0.0,10.67,18.67,28,44.0,4.0,6.0,54.5455
1,3,0,4.5,22.00,31.00,20,22.0,1.0,4.0,100.0000
2,4,1,1.0,14.33,21.33,27,37.0,7.0,4.0,36.3636
3,5,0,0.0,8.67,14.67,29,37.0,4.0,4.0,44.4444
4,6,0,0.5,18.67,25.67,25,30.0,1.0,5.0,83.3333


,RID,Test_data,Age,Gender,Year_education,Ethnicity,Race,Marital_status,High_risk_ApoE4,Ventricles,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,0,74.3,Male,16,Not Hisp/Latino,White,Married,0.0,118233.0,...,0.694,3835,1622,2.077,0.746,15683,4272,3.028,0.649,4396
1,3,0,81.3,Male,18,Not Hisp/Latino,White,Married,1.0,84599.0,...,0.591,3681,1734,1.942,0.696,10387,3316,2.545,0.686,3304
2,4,1,67.5,Male,10,Hisp/Latino,White,Married,0.0,39605.0,...,0.588,4060,1728,2.18,0.607,11156,3598,2.67,0.631,1338
3,5,0,73.7,Male,16,Not Hisp/Latino,White,Married,0.0,34062.0,...,0.628,5180,1868,2.543,0.709,11579,3387,2.911,0.66,1623
4,6,0,80.4,Female,13,Not Hisp/Latino,White,Married,0.0,39826.0,...,0.631,3078,1241,2.141,0.701,9641,2781,2.9,0.727,1035


In [18]:
print(diagnosis_df["Diagnosis"].value_counts(dropna=False))

Diagnosis
LMCI    562
CN      417
AD      342
EMCI    310
SMC     106
Name: count, dtype: int64


# Handling missing values

In [19]:
# Check whether RID is unique in each sheet
for name, df in {
    "Diagnosis": diagnosis_df,
    "Cognitive": cognitive_scores_df,
    "Patient data": data_df
}.items():

    print(
        name,
        "duplicate RID count:",
        df["RID"].duplicated().sum()
    )

Diagnosis duplicate RID count: 0
Cognitive duplicate RID count: 0
Patient data duplicate RID count: 0


In [20]:
# Keep the diagnosis and target information
diagnosis_clean = diagnosis_df[
    ["RID", "Diagnosis", "FDG_PET"]
].copy()


# Remove split indicators from predictor sheets
cognitive_clean = cognitive_scores_df.drop(
    columns=["Test_data"],
    errors="ignore"
).copy()


data_clean = data_df.drop(
    columns=["Test_data"],
    errors="ignore"
).copy()

# Check unique identifiers before merging
assert diagnosis_clean["RID"].is_unique
assert cognitive_clean["RID"].is_unique
assert data_clean["RID"].is_unique

# Merge the datasets
merged_df = pd.merge(
    diagnosis_clean,
    cognitive_clean,
    on="RID",
    how="inner",
    validate="one_to_one"
)

merged_df = pd.merge(
    merged_df,
    data_clean,
    on="RID",
    how="inner",
    validate="one_to_one"
)
print("Merged dataset shape:", merged_df.shape)

display(merged_df.head())


Merged dataset shape: (1737, 370)


,RID,Diagnosis,FDG_PET,CDRSB,ADAS11,ADAS13,MMSE,RAVLT_immediate,RAVLT_learning,RAVLT_forgetting,...,Cortical Thickness Standard Deviation of RightLingual,Volume (Cortical Parcellation) of RightMedialOrbitofrontal,Surface Area of RightMedialOrbitofrontal,Cortical Thickness Average of RightMedialOrbitofrontal,Cortical Thickness Standard Deviation of RightMedialOrbitofrontal,Volume (Cortical Parcellation) of RightMiddleTemporal,Surface Area of RightMiddleTemporal,Cortical Thickness Average of RightMiddleTemporal,Cortical Thickness Standard Deviation of RightMiddleTemporal,Volume (WM Parcellation) of FourthVentricle
0,2,CN,1.36926,0.0,10.67,18.67,28,44.0,4.0,6.0,...,0.694,3835,1622,2.077,0.746,15683,4272,3.028,0.649,4396
1,3,AD,1.09079,4.5,22.00,31.00,20,22.0,1.0,4.0,...,0.591,3681,1734,1.942,0.696,10387,3316,2.545,0.686,3304
2,4,LMCI,NaN,1.0,14.33,21.33,27,37.0,7.0,4.0,...,0.588,4060,1728,2.18,0.607,11156,3598,2.67,0.631,1338
3,5,CN,1.29799,0.0,8.67,14.67,29,37.0,4.0,4.0,...,0.628,5180,1868,2.543,0.709,11579,3387,2.911,0.66,1623
4,6,LMCI,NaN,0.5,18.67,25.67,25,30.0,1.0,5.0,...,0.631,3078,1241,2.141,0.701,9641,2781,2.9,0.727,1035


In [21]:
missing_values = merged_df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing values": missing_values,
    "Missing percentage": (
        missing_values / len(merged_df)
    ) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing percentage",
    ascending=False
)

display(missing_summary.head(20))
print("Total missing values:", merged_df.isnull().sum().sum())

,Missing values,Missing percentage
FDG_PET,434,24.985607
Volume (WM Parcellation) of FifthVentricle,291,16.753022
Fusiform,271,15.601612
MidTemp,271,15.601612
Entorhinal,271,15.601612
Hippocampus,248,14.277490
Ventricles,82,4.720783
WholeBrain,48,2.763385
Intra cranial volume,15,0.863558
ADAS13,14,0.805987


Total missing values: 2003


In [22]:
# Remove records without a target label
merged_df = merged_df.dropna(
    subset=["Diagnosis"]
).copy()

# Clean whitespace
merged_df["Diagnosis"] = (
    merged_df["Diagnosis"].astype(str).str.strip()
)

# Check diagnosis distribution
print(merged_df["Diagnosis"].value_counts())

# Map all five diagnosis categories
category_mapping = {
    "CN": "Cognitively Normal",
    "AD": "Alzheimer's Disease",
    "LMCI": "Late Mild Cognitive Impairment",
    "EMCI": "Early Mild Cognitive Impairment",
    "SMC": "Significant Memory Concern"
}

# Check for unexpected labels
unknown_labels = (
    set(merged_df["Diagnosis"].unique())
    - set(category_mapping.keys())
)

if unknown_labels:
    raise ValueError(
        f"Unexpected diagnosis labels: {unknown_labels}"
    )

# Convert abbreviations into readable labels
merged_df["Diagnosis"] = (
    merged_df["Diagnosis"].map(category_mapping)
)

print("\nUpdated diagnosis distribution:")
print(merged_df["Diagnosis"].value_counts())

Diagnosis
LMCI    562
CN      417
AD      342
EMCI    310
SMC     106
Name: count, dtype: int64

Updated diagnosis distribution:
Diagnosis
Late Mild Cognitive Impairment     562
Cognitively Normal                 417
Alzheimer's Disease                342
Early Mild Cognitive Impairment    310
Significant Memory Concern         106
Name: count, dtype: int64


In [23]:
# Target variable
y = merged_df["Diagnosis"]

# Input features
X = merged_df.drop(
    columns=["Diagnosis", "RID"]
)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (1737, 368)
Target shape: (1737,)

Target distribution:
Diagnosis
Late Mild Cognitive Impairment     562
Cognitively Normal                 417
Alzheimer's Disease                342
Early Mild Cognitive Impairment    310
Significant Memory Concern         106
Name: count, dtype: int64


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training features: (1389, 368)
Testing features: (348, 368)

Training distribution:
Diagnosis
Late Mild Cognitive Impairment     449
Cognitively Normal                 333
Alzheimer's Disease                274
Early Mild Cognitive Impairment    248
Significant Memory Concern          85
Name: count, dtype: int64

Testing distribution:
Diagnosis
Late Mild Cognitive Impairment     113
Cognitively Normal                  84
Alzheimer's Disease                 68
Early Mild Cognitive Impairment     62
Significant Memory Concern          21
Name: count, dtype: int64


In [25]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 19
Number of categorical features: 349

Categorical features:
['Gender', 'Ethnicity', 'Race', 'Marital_status', 'Volume (WM Parcellation) of RightPallidum', 'Volume (Cortical Parcellation) of RightParacentral', 'Surface Area of RightParacentral', 'Cortical Thickness Average of RightParacentral', 'Cortical Thickness Standard Deviation of RightParacentral', 'Volume (Cortical Parcellation) of RightParahippocampal', 'Surface Area of RightParahippocampal', 'Cortical Thickness Average of RightParahippocampal', 'Cortical Thickness Standard Deviation of RightParahippocampal', 'Volume (Cortical Parcellation) of RightParsOpercularis', 'Surface Area of RightParsOpercularis', 'Cortical Thickness Average of RightParsOpercularis', 'Cortical Thickness Standard Deviation of RightParsOpercularis', 'Volume (Cortical Parcellation) of RightParsOrbitalis', 'Surface Area of RightParsOrbitalis', 'Cortical Thickness Average of RightParsOrbitalis', 'Cortical Thickness Standard Dev

# Build preprocessing pipeline

In [26]:
# Numerical preprocessing
numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [29]:
rf_model=RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Combine preprocessing and model

In [31]:
model=Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("classifier",rf_model)
    ]
)